# 05 — Evaluate

Read-only comparison of the latest complete model executions recorded in MLflow. This stage reports selected-candidate cross-validation and sealed-test metrics without ranking models or writing any artifacts.

**Inputs:** MLflow run metadata and metrics only  
**Outputs:** in-memory comparison tables and figures only

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
## Setup

import mlflow
from IPython.display import Markdown, display

from src.config import MLFLOW_TRACKING_URI
from src.evaluation import (
    load_latest_complete_comparison_metrics,
    load_latest_complete_feature_subset_cv_metrics,
)
from src.plots import (
    aggregate_comparison_figure,
    alarm_threshold_comparison_figure,
    alarm_threshold_horizon_figure,
    feature_subset_best_comparison_figure,
    feature_subset_candidate_distribution_figure,
    horizon_comparison_figure,
    quartile_comparison_figure,
    quartile_horizon_small_multiples_figure,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## Load the latest complete executions

For each expected experiment, the query inspects finished sealed-test runs newest-first and uses the first execution whose sealed metrics and uniquely matched selected CV parent are complete. It warns and falls back when a newer execution is incomplete, and warns and skips models with no complete execution. All included models must share the same train/test artifact hashes and forecast horizon.

This notebook never starts or logs an MLflow run, reads a feature/test artifact, loads a model, or reruns training.

In [ ]:
comparison_metrics_df = load_latest_complete_comparison_metrics()

run_summary_df = (
    comparison_metrics_df[
        [
            "model",
            "phase",
            "run_id",
            "execution_uuid",
            "completed_at_utc",
            "train_input_sha256",
            "test_input_sha256",
            "forecast_horizon_hours",
            "cohort_row_count",
            "cohort_size_differs",
        ]
    ]
    .drop_duplicates()
    .sort_values(["model", "phase"], kind="stable")
    .reset_index(drop=True)
)
display(run_summary_df)
display(comparison_metrics_df)

## Cohort comparability

The fitted tabular models and persistence baseline use the common eligible cohort. The RNN remains in this descriptive comparison, but its lookback-sequence eligibility can remove additional issue times. The table and note below make that difference explicit; the plots do not imply a like-for-like ranking.

In [ ]:
cohort_summary_df = (
    comparison_metrics_df[["model", "cohort_row_count", "cohort_size_differs"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
display(cohort_summary_df)

rnn_cohort = cohort_summary_df.loc[cohort_summary_df["model"].eq("RNN")]
if not rnn_cohort.empty:
    common_cohort_rows = int(cohort_summary_df["cohort_row_count"].max())
    rnn_cohort_rows = int(rnn_cohort["cohort_row_count"].iloc[0])
    display(
        Markdown(
            "**RNN cohort caveat:** RNN metrics retain "
            f"{rnn_cohort_rows:,} sequence-eligible issue times versus "
            f"{common_cohort_rows:,} for the largest common cohort "
            f"({common_cohort_rows - rnn_cohort_rows:,} fewer). "
            "Cross-model figures are descriptive, not a ranking."
        )
    )

## Cross-validation comparisons

The selected candidate for each model is shown as its logged mean ± standard deviation.

### Aggregate cross-validation metrics

This plot compares MAE, RMSE, ME, and R² aggregated across all forecast horizons. Each marker is the selected candidate's mean across validation folds, and the error bar shows one logged standard deviation.

In [ ]:
cv_aggregate_figure = aggregate_comparison_figure(
    comparison_metrics_df, phase="cross_validation"
)
display(cv_aggregate_figure)

### Cross-validation metrics by forecast horizon

This plot shows the four metrics separately for every forecast horizon, making changes in accuracy, bias, and explained variance visible as lead time increases. Error bars show the logged fold-to-fold standard deviation.

In [ ]:
cv_horizon_figure = horizon_comparison_figure(
    comparison_metrics_df, phase="cross_validation"
)
display(cv_horizon_figure)

## Best candidate within each feature subset

For every subset-aware model, this section takes all candidate-parent runs from the same latest complete execution used above and re-applies that estimator's configured CV selection metric and deterministic simplicity tie-breakers within each feature subset. Each model point is the winning candidate's aggregate mean across validation folds; error bars show one fold-to-fold standard deviation. The dashed persistence line repeats the same feature-independent baseline across the subsets for comparison, with its error bars showing the logged fold-to-fold standard deviation. RNN is excluded because it does not search the shared named feature subsets.

This is a controlled cross-validation ablation, not a sealed-test comparison. The training notebooks establish a common eligible cohort and identical folds for all subsets within a model execution.

In [ ]:
feature_subset_metrics_df = load_latest_complete_feature_subset_cv_metrics()

feature_subset_candidate_counts_df = (
    feature_subset_metrics_df.groupby(["model", "subset"], as_index=False)
    .agg(
        candidate_count=("run_id", "size"),
        best_candidate_count=("is_best_within_subset", "sum"),
    )
    .sort_values(["model", "subset"], kind="stable")
    .reset_index(drop=True)
)
display(feature_subset_candidate_counts_df)

feature_subset_best_figure = feature_subset_best_comparison_figure(
    feature_subset_metrics_df, comparison_metrics_df
)
display(feature_subset_best_figure)

## All candidates by feature subset

These grouped box plots retain every hyperparameter candidate instead of only the within-subset winner. Each point is one candidate's aggregate metric mean across validation folds; the boxes summarize the distribution across candidates for that model and subset. The dashed persistence line repeats the same feature-independent aggregate CV baseline across every subset for comparison; it is not part of the candidate distributions. The boxes show search sensitivity and robustness, not fold-level uncertainty. Candidate counts differ between model families because their search spaces differ, so box widths and apparent density must not be interpreted as equal search budgets.

In [ ]:
feature_subset_distribution_figure = feature_subset_candidate_distribution_figure(
    feature_subset_metrics_df, comparison_metrics_df
)
display(feature_subset_distribution_figure)

## Sealed-test comparisons

Sealed-test panels show point estimates on data that was not used for candidate selection.

### Aggregate sealed-test metrics

This plot compares each selected model's MAE, RMSE, ME, and R² after aggregating predictions across all forecast horizons. The markers are single sealed-test point estimates, so no uncertainty bars are shown.

In [ ]:
test_aggregate_figure = aggregate_comparison_figure(
    comparison_metrics_df, phase="sealed_test"
)
display(test_aggregate_figure)

### Sealed-test metrics by forecast horizon

This plot breaks the sealed-test metrics down by forecast horizon. The lines show how each model's test accuracy, bias, and explained variance change from the shortest to the longest lead time.

In [ ]:
test_horizon_figure = horizon_comparison_figure(
    comparison_metrics_df, phase="sealed_test"
)
display(test_horizon_figure)

## Sealed-test water-level regime diagnostics

These descriptive diagnostics are computed only from sealed-test forecast values. The Q1–Q4 cutoffs are derived from finite, non-imputed target-station training observations and are shown in the compact table below. Alarm membership is a separate overlapping subset at the configured PegelAlarm `defaultAlarmValueCm` snapshot (545 cm for station 207241-at), not an official flood threshold. Regime rows never affect candidate selection or model ranking.

Counts are scored forecast values, so they can overlap between horizons and between Q4 and Alarm. Empty cells retain count zero and display unavailable metrics as gaps.

In [ ]:
regime_summary_df = (
    comparison_metrics_df.loc[
        comparison_metrics_df["phase"].isin(
            ["sealed_test_quartile", "sealed_test_alarm"]
        )
        & comparison_metrics_df["scope"].eq("aggregate")
    ][
        [
            "model",
            "phase",
            "regime",
            "lower_bound_cm",
            "upper_bound_cm",
            "scored_values",
            "metric",
            "value",
        ]
    ]
    .sort_values(["phase", "regime", "model", "metric"], kind="stable")
    .reset_index(drop=True)
)
display(regime_summary_df)

alarm_counts_df = regime_summary_df.loc[
    regime_summary_df["phase"].eq("sealed_test_alarm")
]["scored_values"].drop_duplicates()
if not alarm_counts_df.empty and alarm_counts_df.eq(0).all():
    display(
        Markdown(
            "**Alarm-level performance cannot be assessed:** no sealed-test forecast values reached 545 cm."
        )
    )

In [ ]:
quartile_aggregate_figure = quartile_comparison_figure(comparison_metrics_df)
display(quartile_aggregate_figure)

In [ ]:
for regime_metric in ("mae", "rmse", "me"):
    display(
        quartile_horizon_small_multiples_figure(
            comparison_metrics_df, metric=regime_metric
        )
    )

In [ ]:
alarm_aggregate_figure = alarm_threshold_comparison_figure(comparison_metrics_df)
display(alarm_aggregate_figure)
for regime_metric in ("mae", "rmse", "me"):
    display(alarm_threshold_horizon_figure(comparison_metrics_df, metric=regime_metric))